# 01 · Monitoring Agent

**Role in the pipeline:** first stage. Scan every account each night and flag those whose behaviour has shifted.
It does **not** judge risk — it decides *who the analyst should look at*.

This notebook lets you:
1. Load the Gold-layer customer data
2. Inspect and tune the monitoring rules
3. Run the agent and review who was flagged and why
4. Hand the flagged list to the Analyst notebook (`_dev/flagged.json`)

Run cells top to bottom (**Run All** in VS Code).

In [ ]:
# --- Setup: point the notebook at the project's `copilot` package ---------
# Must run BEFORE any `copilot` import: settings are read once from env vars.
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "copilot").exists():          # works from notebooks/ or project root
    if ROOT == ROOT.parent:
        raise RuntimeError("Open the cs_ai_copilot_dss folder in VS Code and run from there.")
    ROOT = ROOT.parent
os.chdir(ROOT)                                            # playbook path is relative to project root
sys.path.insert(0, str(ROOT / "src"))

DEV = ROOT / "notebooks" / "_dev"                         # local, git-ignorable sandbox
DEV.mkdir(parents=True, exist_ok=True)
os.environ["DATABASE_URL"] = f"sqlite:///{(DEV / 'copilot_dev.db').as_posix()}"
os.environ["MODEL_DIR"] = str(DEV / "models")
os.environ.pop("CHROMA_HOST", None)                       # notebooks use the in-memory RAG store

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Project root:", ROOT)
print("Dev database:", os.environ["DATABASE_URL"])

## 1 · Load Gold-layer data
Synthetic B2B textile dataset. Replace `load_gold()` with your Databricks / Postgres reader later — the column names are the contract.

In [ ]:
from copilot.data import load_gold, FEATURES

gold = load_gold()
print(f"{len(gold):,} accounts · {len(FEATURES)} model features")
gold[["account_id", "name", "segment", "supplier_origin"] + FEATURES[:6]].head()

In [ ]:
gold[["recency_days", "order_volume_change_pct", "late_payments_12m",
      "support_tickets_90d", "quality_complaints_12m", "engagement_score"]].describe().round(2)

## 2 · Monitoring rules
Each rule is `(column, operator, threshold, signal label)`. These live in `src/copilot/agents/monitoring.py` → `RULES`. Edit there to change what production uses; edit the copy below to experiment.

In [ ]:
from copilot.agents.monitoring import MonitoringAgent, RULES

rules = list(RULES)          # experiment on a copy
pd.DataFrame(rules, columns=["column", "op", "threshold", "signal"])

### How often does each rule fire on its own?
A rule that fires for half the customer base is noise; one that never fires is dead weight.

In [ ]:
from copilot.agents.monitoring import _OPS

hit_rates = {label: _OPS[op](gold[col], thr).mean() for col, op, thr, label in rules}
hit_rates = pd.Series(hit_rates).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(hit_rates.index, hit_rates.values * 100, color="#3b6ea8")
for i, v in enumerate(hit_rates.values):
    ax.text(v * 100 + 0.5, i, f"{v:.0%}", va="center", fontsize=9)
ax.set_xlabel("% of accounts triggering the rule")
ax.set_title("Monitoring rule hit rates")
plt.tight_layout(); plt.show()

## 3 · Choose `min_signals`
An account is flagged when it trips at least `min_signals` rules. The trade-off: more flags = more analyst work and a longer review queue.

In [ ]:
sweep = pd.DataFrame([
    {"min_signals": k, "flagged": len(MonitoringAgent(rules, k).run(gold).flagged)}
    for k in range(1, 5)])
sweep["% of base"] = (sweep.flagged / len(gold) * 100).round(1)

# What share of accounts that actually churned would each setting catch?
churned = set(gold.loc[gold.churned_next_6m == 1, "account_id"])
sweep["% of churners caught"] = [
    round(len(churned & set(MonitoringAgent(rules, k).run(gold).flagged)) / len(churned) * 100, 1)
    for k in sweep.min_signals]
sweep

> The churn column is only available because the data is synthetic — with real data you'd evaluate this after the pilot quarter. Production default is **2**.

In [ ]:
MIN_SIGNALS = 2          # <- tune here

agent = MonitoringAgent(rules, min_signals=MIN_SIGNALS)
result = agent.run(gold)
print(f"Flagged {len(result.flagged)} of {len(gold)} accounts ({len(result.flagged)/len(gold):.1%})")

## 4 · Who was flagged, and why

In [ ]:
flagged_df = (gold.set_index("account_id")
              .loc[result.account_ids, ["name", "segment", "supplier_origin", "avg_order_value"]]
              .assign(signals=[result.flagged[a] for a in result.account_ids],
                      n_signals=[len(result.flagged[a]) for a in result.account_ids])
              .sort_values(["n_signals", "avg_order_value"], ascending=False))
flagged_df.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
sig_counts = pd.Series([s for sigs in result.flagged.values() for s in sigs]).value_counts()
axes[0].barh(sig_counts.index[::-1], sig_counts.values[::-1], color="#3b6ea8")
axes[0].set_title("Signals among flagged accounts")
seg = flagged_df.segment.value_counts()
axes[1].bar(seg.index, seg.values, color="#c0703b")
axes[1].set_title("Flagged accounts by RFM segment")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 5 · Hand-off to the Analyst agent

In [ ]:
handoff = {"min_signals": MIN_SIGNALS, "flagged": result.flagged}
(DEV / "flagged.json").write_text(json.dumps(handoff, indent=1))
print(f"Saved {len(result.flagged)} flagged accounts → {DEV / 'flagged.json'}")
print("Next: open 02_analyst_agent.ipynb")